In [1]:
import pandas as pd
import numpy as np
from scipy import stats

In [2]:
df=pd.read_csv('raw_data/multi_stations_dataset.csv')

In [3]:
# Convert the first column to datetime and rename it
df['Timestamp'] = pd.to_datetime(df.iloc[:, 0])
df = df.drop(columns=[df.columns[0]])

In [4]:
# Handle duplicates: If multiple entries exist for the same hour, take the mean
df = df.groupby('Timestamp').mean().reset_index()

# 3. Outlier Removal
# We identified that 'load' has extreme values (e.g., 0 or 99991)
# We use Z-score to replace values > 3 standard deviations with NaN
def remove_outliers(series):
    z_scores = np.abs(stats.zscore(series.dropna()))
    # Create a mask for outliers
    outliers = series.copy()
    # Map back the z-scores to the original index
    z_map = pd.Series(z_scores, index=series.dropna().index)
    outliers[z_map[z_map > 3].index] = np.nan
    return outliers

df['load'] = remove_outliers(df['load'])

# 4. Consistent Rows (Reindexing)
# Create a complete hourly range from the very start to the end
full_range = pd.date_range(start=df['Timestamp'].min().replace(hour=0),
                           end=df['Timestamp'].max(),
                           freq='H')
df = df.set_index('Timestamp').reindex(full_range)
df.index.name = 'Timestamp'

# 5. Missing Data Interpolation
# Use linear interpolation to fill gaps created by reindexing and outlier removal
df = df.interpolate(method='linear', limit_direction='both')

# 6. Feature Engineering
# Time Features
df['Hour'] = df.index.hour
df['DayOfWeek'] = df.index.dayofweek
df['Month'] = df.index.month
df['Weekend'] = df['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)

# Cyclical Encoding (helps model understand time loops)
df['Hour_sin'] = np.sin(2 * np.pi * df['Hour'] / 24)
df['Hour_cos'] = np.cos(2 * np.pi * df['Hour'] / 24)
df['Day_sin'] = np.sin(2 * np.pi * df['DayOfWeek'] / 7)
df['Day_cos'] = np.cos(2 * np.pi * df['DayOfWeek'] / 7)
df['Month_sin'] = np.sin(2 * np.pi * df['Month'] / 12)
df['Month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)

# Degree Hours (CDH/HDH) - Base 18°C
stations = ['Boston', 'Hartford', 'Providence', 'Portland', 'Burlington']
for city in stations:
    temp_col = f'Temp_{city}'
    df[f'CDH_{city}'] = (df[temp_col] - 18).clip(lower=0)
    df[f'HDH_{city}'] = (18 - df[temp_col]).clip(lower=0)

# Rolling Statistics
df['rolling_24'] = df['load'].rolling(window=24, min_periods=1).mean()
df['rolling_168'] = df['load'].rolling(window=168, min_periods=1).mean()

C:\Users\admin\AppData\Local\Temp\ipykernel_11364\2076239330.py:20: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  full_range = pd.date_range(start=df['Timestamp'].min().replace(hour=0),


In [5]:
# 1. Standardize Timestamp format
df = df.reset_index() # Convert the Timestamp index to a column

# 2. Re-calculate Circular Features (Trigonometric Encoding)
# preprocessed_load_data.csv uses (Month - 1) to start January at 0
df['Month_sin'] = np.sin(2 * np.pi * (df['Month'] - 1) / 12)
df['Month_cos'] = np.cos(2 * np.pi * (df['Month'] - 1) / 12)

# Hour encoding (standard 24-hour cycle)
df['Hour_sin'] = np.sin(2 * np.pi * df['Hour'] / 24)
df['Hour_cos'] = np.cos(2 * np.pi * df['Hour'] / 24)

# Day of Week encoding (if applicable, using DayOfWeek 0-6)
df['Day_sin'] = np.sin(2 * np.pi * df['DayOfWeek'] / 7)
df['Day_cos'] = np.cos(2 * np.pi * df['DayOfWeek'] / 7)

# 3. Recalculate CDH and HDH (Cooling/Heating Degree Hours)
# Base temperature appears to be 18 degrees Celsius
temp_cols = ['Boston', 'Hartford', 'Providence', 'Portland', 'Burlington']
for city in temp_cols:
    df[f'CDH_{city}'] = (df[f'Temp_{city}'] - 18).clip(lower=0)
    df[f'HDH_{city}'] = (18 - df[f'Temp_{city}']).clip(lower=0)

# 4. Correct Data Types
df['Holiday'] = df['Holiday'].astype(int)

# 5. Rounding to Match Precision
# preprocessed_load_data.csv often uses 5 to 10 decimal places
cols_to_round = df.select_dtypes(include=[np.number]).columns
df[cols_to_round] = df[cols_to_round].round(10)

# 6. Rolling Features
# Note: If values still differ, it is because preprocessed_load_data.csv
# was likely calculated using a longer historical window before slicing.
df['rolling_24'] = df['load'].rolling(window=24, min_periods=1).mean()
df['rolling_168'] = df['load'].rolling(window=168, min_periods=1).mean()

# Save the corrected version
df.to_csv('preprocessed_load_data.csv', index=False)